In [22]:
import os
import numpy as np
from pathlib import Path
import shutil, subprocess
import matplotlib.pyplot as plt

# Simulation & Geometry Tools
from lammps import lammps
from ase import Atoms
from ase.io import read, write
from ase.geometry import find_mic
from scipy.spatial import cKDTree

# --- Directory Management ---
base_dir = Path.cwd().parent
input_dir = base_dir / "input_raw"
output_dir = base_dir / "input"

# Clear existing input directory to prevent file conflicts
if input_dir.exists():
    shutil.rmtree(input_dir)

# Recreate fresh directories
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

potential_file = base_dir / "potentials/mendelev03.fs"
print(f"Directory cleared. Working in: {input_dir}")
print(f"Potential:  {potential_file}")
print(f"Output dir: {output_dir}")

Directory cleared. Working in: /home/Ethan/Projects/neb_2/input_raw
Potential:  /home/Ethan/Projects/neb_2/potentials/mendelev03.fs
Output dir: /home/Ethan/Projects/neb_2/input


In [23]:
# --- Configuration ---
LAT_PARAM = 2.8665
BURG_MAG = LAT_PARAM*np.sqrt(3)/2
X_SIZE = 43
Y_SIZE = 75
Z_SIZE = 6

# --- Execution ---
print(f"Generating 110 structures in: {input_dir}")

unitcell_path = input_dir / f"slab.lmp"
final_output = input_dir / f"Fe_S111_110_raw.lmp"

final_output.unlink(missing_ok=True)

top_deform = -0.5 / (X_SIZE + 1)
bot_deform = 0.5 / X_SIZE

subprocess.run(["atomsk", "--create", "bcc", str(LAT_PARAM), "Fe", 
                "orient", "[11-2]", "[1-10]", "[111]", 
                "-duplicate", str(X_SIZE), str(Y_SIZE), str(Z_SIZE), 
                str(unitcell_path)], 
                check=True, capture_output=True)

subprocess.run(["atomsk", str(unitcell_path), 
                "-dislocation", "0.51*box", "0.501*box",
                "screw", "Z", "Y", str(BURG_MAG),
                "-cell", "add", str(BURG_MAG/2), "zx",
                "-alignx", str(final_output)])

for tmp_file in [unitcell_path]:
    tmp_file.unlink(missing_ok=True)

print("\n110 configurations generated.")

Generating 110 structures in: /home/Ethan/Projects/neb_2/input_raw
 ___________________________________________________
|              ___________                          |
|     o---o    A T O M S K                          |
|    o---o|    Version master-2025-10-28 (Beta)     |
|    |   |o    (C) 2010 Pierre Hirel                |
|    o---o     https://atomsk.univ-lille.fr         |
|___________________________________________________|
*** Working out of office hours? You should sleep sometimes. :-)
>>> Opening input file: /home/Ethan/Projects/neb_2/input_raw/slab.lmp (8.9 MB)
..> Input file was read successfully (116100 atoms).
>>> Inserting a screw dislocation with line along Z,
    conserving the total number of atoms,
    b=[0.000 0.000 2.482] at (153.981,152.323)
..> Dislocation stresses were computed.
..> Prelogarithmic energy factor: b²/4pi = 0.49040545
..> One dislocation was successfully inserted.
>>> Adding 1.241 A to the ZX tilt...
..> Cell vector was modified.
>>> Align

In [24]:
TOTAL_Y_LAYERS  = int(Y_SIZE)   # Layers in Y direction
NUM_FIXED_LAYERS = 6

atoms = read(str(final_output), format="lammps-data", atom_style="atomic")

# 1. Sort atoms: X → Y → Z for consistency
pos      = atoms.get_positions()
sort_idx = np.lexsort((pos[:, 2], pos[:, 1], pos[:, 0]))
atoms    = atoms[sort_idx].copy()
atoms.center()

# 2. Identify fixed boundary layers (Split into Top and Bottom)
y_coords = atoms.get_positions()[:, 1]
y_min, y_max = y_coords.min(), y_coords.max()
y_height = y_max - y_min

layer_thickness    = y_height / TOTAL_Y_LAYERS
boundary_threshold = layer_thickness * (NUM_FIXED_LAYERS / 2) # Adjust if needed

# Define masks for top and bottom specifically
bottom_mask = (y_coords <= y_min + boundary_threshold + 0.01)
top_mask    = (y_coords >= y_max - boundary_threshold - 0.01)

# 3. Assign types
# Default is Type 1 (Mobile)
new_types = np.ones(len(atoms), dtype=int)

# Assign Type 2 to Top, Type 3 to Bottom
new_types[top_mask] = 2
new_types[bottom_mask] = 3

atoms.set_array('type', new_types)

# Update chemical symbols for visualization (Fe, Ru, Ni or similar)
# Type 1: Fe, Type 2: Ru (Top), Type 3: Ni (Bottom)
sym_map = {1: 'Fe', 2: 'Ru', 3: 'Ni'}
atoms.set_chemical_symbols([sym_map[t] for t in new_types])

atoms.set_pbc([True, False, True])

# 4. Write
out_path = output_dir / f"typed_{final_output.name}"
# Note: specorder must match the number of types (1, 2, 3)
write(str(out_path), atoms, format="lammps-data",
        specorder=["Fe", "Fe", "Fe"], atom_style="atomic")

n_mobile = int(np.sum(new_types == 1))
n_top    = int(np.sum(new_types == 2))
n_bot    = int(np.sum(new_types == 3))

print(f"{out_path.name}:")
print(f"  - {n_mobile} Mobile (Type 1)")
print(f"  - {n_top} Top Fixed (Type 2)")
print(f"  - {n_bot} Bottom Fixed (Type 3)")

typed_Fe_S111_110_raw.lmp:
  - 106812 Mobile (Type 1)
  - 4644 Top Fixed (Type 2)
  - 4644 Bottom Fixed (Type 3)
